In [1]:
# CELL 1 — SABSE PEHLE YE RUN KARO
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Data load karo (apna exact path daal do)
df = pd.read_csv('CVD_cleaned.csv')

# Column names check karo ek baar
print("Columns:", df.columns.tolist())
print("\nTarget distribution:")
print(df['Heart_Disease'].value_counts())

Columns: ['General_Health', 'Checkup', 'Exercise', 'Heart_Disease', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Height_(cm)', 'Weight_(kg)', 'BMI', 'Smoking_History', 'Alcohol_Consumption', 'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption']

Target distribution:
Heart_Disease
No     283883
Yes     24971
Name: count, dtype: int64


In [2]:
# CELL 2
# Target aur features alag karo (exact column name daal do)
# Sabse common names yeh hain — jo match kare wo use karo
target_column = 'Heart_Disease'  # ya 'Heart_Attack_Risk' ya 'Heart Disease'

y = df[target_column]
X = df.drop(target_column, axis=1)

# Categorical columns ko one-hot encode
X = pd.get_dummies(X, drop_first=True)

# Train-test split (stratify = class balance maintain karega)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data preparation complete!")
print(f"Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}")

Data preparation complete!
Train shape: (247083, 37), Test shape: (61771, 37)


In [3]:
# CELL 3 — BEST WORKING MODEL
# Class imbalance handle karne ke liye scale_pos_weight
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    scale_pos_weight=scale_weight,
    eval_metric='auc',
    random_state=42,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8
)

model.fit(X_train_scaled, y_train)

# Results
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['No' 'Yes']